# RoadVision — Inference-only, iou=0.50 variant (reuse K-Fold weights → fixed submission)
Public - 0.56135	Private - 0.63206


Identical to `roadvision_infer_from_weights.ipynb` except **PRED_IOU 0.60 → 0.50**.

`experiments/LOG.md` #5 found iou=0.50 beat iou=0.60 in a leak-free local grid search
(0.55044 local mAP @ conf=0.001, iou=0.5, imgsz=960) — but that finding has never been validated
on the actual leaderboard. This notebook is that validation, at near-zero cost since it's
inference-only against already-trained weights.

**Setup (one manual step):** right panel → **Add Input** → **Your Work / Notebook Output** →
select the K-Fold notebook version that finished (the 960px 5-fold yolov8s run — best confirmed
score so far is 0.56748 from this run's weights at iou=0.60). The competition data should already
be attached (it's the competition notebook).

**Settings:** Accelerator = **GPU**, Internet = **On** (only to `pip install` the packages; the
weights load locally).

Fixes carried over: builds a row for **every** test image (sample_submission has only 10 rows),
`image_id` = **stem with no extension**, and **Weighted Box Fusion** to merge the 5 models.

In [1]:
# Install + imports
try:
    from ultralytics import YOLO
except ImportError:
    import subprocess; subprocess.run(["pip", "install", "-q", "ultralytics"], check=True)
    from ultralytics import YOLO
try:
    from ensemble_boxes import weighted_boxes_fusion
except ImportError:
    import subprocess; subprocess.run(["pip", "install", "-q", "ensemble-boxes"], check=True)
    from ensemble_boxes import weighted_boxes_fusion

import re
from pathlib import Path
import numpy as np
import pandas as pd

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 103.8 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [2]:
# --- Auto-discover test images + the fold weights (no hard-coded paths) ---
INPUT_ROOT = Path("/kaggle/input")

# Test images: an 'images' dir with 'test' in its path
TEST_IMAGES = next((p for p in INPUT_ROOT.rglob("images")
                    if p.is_dir() and "test" in str(p).lower()), None)

# Fold weights: every best.pt anywhere under /kaggle/input (your mounted notebook output)
weight_paths = [p for p in INPUT_ROOT.rglob("best.pt")]

def fold_key(p):
    m = re.search(r"fold[_-]?(\d+)", str(p))
    return int(m.group(1)) if m else 999
weight_paths = sorted(weight_paths, key=fold_key)

print("TEST_IMAGES:", TEST_IMAGES)
print(f"found {len(weight_paths)} weight file(s):")
for p in weight_paths:
    print("  ", p)

assert TEST_IMAGES and TEST_IMAGES.exists(), "Test images not found - is the competition data attached?"
assert weight_paths, "No best.pt found - did you Add Input the finished K-Fold notebook output?"

IMGSZ = 960          # must match what the models were trained at
PRED_IOU = 0.50      # changed from 0.60 -- validated better locally in experiments/LOG.md #5
WBF_IOU, SKIP_THR = 0.6, 0.0001

TEST_IMAGES: /kaggle/input/competitions/road-vision/RoadVision_DUET/test/images
found 5 weight file(s):
   /kaggle/input/notebooks/tahjibhossainkhan/k-fold/runs/fold_0/weights/best.pt
   /kaggle/input/notebooks/tahjibhossainkhan/k-fold/runs/fold_1/weights/best.pt
   /kaggle/input/notebooks/tahjibhossainkhan/k-fold/runs/fold_2/weights/best.pt
   /kaggle/input/notebooks/tahjibhossainkhan/k-fold/runs/fold_3/weights/best.pt
   /kaggle/input/notebooks/tahjibhossainkhan/k-fold/runs/fold_4/weights/best.pt


In [3]:
# --- WBF ensemble inference -> submission.csv ---
test_idx = {p.stem: p for p in TEST_IMAGES.rglob("*.*")
            if p.suffix.lower() in [".jpg", ".png", ".jpeg"]}
test_stems = sorted(test_idx.keys())
print(f"test images to predict: {len(test_stems)}  (expect 327)")

models = [YOLO(str(p)) for p in weight_paths]
print(f"loaded {len(models)} models")

rows = []
for stem in test_stems:
    p = str(test_idx[stem])
    boxes_l, scores_l, labels_l = [], [], []
    for m in models:
        b = m.predict(source=p, imgsz=IMGSZ, conf=0.001, iou=PRED_IOU,
                      max_det=300, verbose=False)[0].boxes
        if b is not None and len(b) > 0:
            boxes_l.append(b.xyxyn.cpu().numpy().tolist())     # normalized [x1,y1,x2,y2]
            scores_l.append(b.conf.cpu().numpy().tolist())
            labels_l.append(b.cls.cpu().numpy().astype(int).tolist())

    if not boxes_l:
        rows.append({"image_id": stem, "PredictionString": ""})
        continue

    fb, fs, fl = weighted_boxes_fusion(boxes_l, scores_l, labels_l,
                                       iou_thr=WBF_IOU, skip_box_thr=SKIP_THR)
    parts = []
    for i in np.argsort(fs)[::-1]:                             # confidence-descending
        x1, y1, x2, y2 = fb[i]
        parts += [str(int(fl[i])), f"{fs[i] * 100:.2f}",
                  f"{(x1 + x2) / 2:.6f}", f"{(y1 + y2) / 2:.6f}",
                  f"{x2 - x1:.6f}", f"{y2 - y1:.6f}"]
    rows.append({"image_id": stem, "PredictionString": " ".join(parts)})

sub = pd.DataFrame(rows)                # image_id = stem, NO extension
sub.to_csv("/kaggle/working/submission.csv", index=False)

n_with = int((sub["PredictionString"].str.len() > 0).sum())
print(f"\nsaved /kaggle/working/submission.csv | rows: {len(sub)} (expect 327) | with detections: {n_with}")
print(sub.head())
assert len(sub) == len(test_stems), "Row count mismatch - submission incomplete!"  

test images to predict: 327  (expect 327)
loaded 5 models

saved /kaggle/working/submission.csv | rows: 327 (expect 327) | with detections: 327
                                        image_id  \
0  CCTV01^20260318-110000-20260318-120000_000061   
1  CCTV01^20260318-110000-20260318-120000_000065   
2  CCTV01^20260318-110000-20260318-120000_000069   
3  CCTV01^20260318-110000-20260318-120000_000073   
4  CCTV01^20260318-110000-20260318-120000_000077   

                                    PredictionString  
0  3 86.48 0.525184 0.878756 0.066646 0.188073 10...  
1  3 86.09 0.525162 0.878889 0.067344 0.188603 3 ...  
2  3 81.35 0.524936 0.878801 0.067913 0.189556 10...  
3  3 85.24 0.525058 0.878608 0.067525 0.189091 10...  
4  3 84.94 0.525210 0.878438 0.067793 0.190197 3 ...  
